In [ ]:
# Chapter 15: Processing Sequences Using RNNs and CNNs

## Global Imports

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

## Recurrent Neurons and Layers
Recurrent Neural Networks (RNNs) are designed to process sequences of arbitrary length by maintaining a "memory" of past inputs. Unlike feedforward networks, recurrent neurons have connections pointing backward. At each time step $t$, a recurrent neuron receives the input $x_{(t)}$ and its own output from the previous time step, $y_{(t-1)}$.

<p align="left"><img src="../fig/figure15.1.png" width="45%"></p>
<p align="left"><img src="../fig/figure15.2.png" width="45%"></p>

### Unrolling Through Time
An RNN can be visualized as unrolling the network through time. At each time step, it processes the current input and the hidden state from the previous step.
- Memory Cell: A part of a neural network that preserves some state across time steps.
- Layers: A layer of recurrent neurons takes a batch of input sequences ($X$) and outputs a batch of sequences ($Y$).

Simple RNN in Keras The simplest RNN layer in Keras is SimpleRNN. It is fully connected and uses a hyperbolic tangent (tanh) activation function by default.

In [2]:
# A simple RNN with one layer and one neuron
# Input shape: [batch_size, time_steps, input_dimensions]
model = keras.models.Sequential([
    keras.layers.SimpleRNN(1, input_shape=[None, 1])
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## Input and Output Sequences
RNNs can handle various input/output configurations:
- Sequence-to-Sequence: Inputs a sequence, outputs a sequence (e.g., stock price prediction).
- Sequence-to-Vector: Inputs a sequence, outputs a single vector (e.g., sentiment analysis).
- Vector-to-Sequence: Inputs a single vector, outputs a sequence (e.g., image captioning).
- Encoder-Decoder: A sequence-to-vector network (encoder) followed by a vector-to-sequence network (decoder) (e.g., language translation).

<p align="left"><img src="../fig/figure15.4.png" width="45%"></p>


## Training RNNs (Backpropagation Through Time)
Training an RNN involves unrolling it through time and using regular backpropagation. This is called Backpropagation Through Time (BPTT).
1. Forward pass: The sequence flows through the unrolled network.
2. Loss calculation: The output sequence is evaluated using a loss function (e.g., MSE) at each time step (or just the last one).
3. Backward pass: Gradients are propagated backward through time (from the last time step to the first).
4. Update: Weights are updated.

<p align="left"><img src="../fig/figure15.5.png" width="45%"></p>

## Forecasting a Time Series
The chapter uses a synthetic time series dataset to demonstrate forecasting.

<p align="left"><img src="../fig/figure15.6.png" width="45%"></p>

### Baseline Metrics
Before building complex models, establish baselines:
- Naive Forecasting: Predict the last observed value ($\hat{y}_{t} = y_{t-1}$).
- Linear Regression: Use a simple Dense layer.

In [3]:
# Linear Model (Baseline)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[50, 1]),
    keras.layers.Dense(1)
])
model.compile(loss="mse", optimizer="adam")
# history = model.fit(X_train, y_train, epochs=20, validation_data=(X_valid, y_valid))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Implementing a Simple RNN
A simple RNN can process the sequence and predict the next step.

In [4]:
model = keras.models.Sequential([
    keras.layers.SimpleRNN(1, input_shape=[None, 1])
])

### Deep RNNs
To solve complex problems, we stack multiple recurrent layers. In Keras, you must set return_sequences=True for all recurrent layers except the last one (unless the last one is also supposed to output a sequence).

<p align="left"><img src="../fig/figure15.7.png" width="45%"></p>

In [5]:
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.SimpleRNN(1)
])

## Forecasting Several Steps Ahead
There are two main approaches to predict multiple future steps (e.g., next 10 values):

<p align="left"><img src="../fig/figure15.8.png" width="45%"></p>

### Option 1: Recursive Prediction
Train the model to predict 1 step ahead, then feed the prediction back as input to predict the next step, and so on. This error accumulates over time.

In [6]:
# Example logic (pseudo-code using the model trained above)
# series = ... # initial data
# for step in range(10):
#     y_pred = model.predict(series[..., np.newaxis])
#     series = np.append(series, y_pred)

### Option 2: Predict All Steps at Once (Seq2Vector)
Train the RNN to output a vector of 10 values at the final step.

In [7]:
# Output layer has 10 units for 10 future steps
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(10)
])

### Option 3: Sequence-to-Sequence (Seq2Seq)
Train the RNN to predict the next 10 steps at every time step. This provides more gradients for training and stabilizes learning. TimeDistributed wrapper is used to apply a Dense layer to every time step independently.

In [8]:
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

## Handling Long Sequences
Simple RNNs suffer from the vanishing gradient problem and forget early inputs in long sequences.

### Layer Normalization
Batch Normalization doesn't work well with RNNs. Layer Normalization is preferred. It normalizes across the features dimension for each instance independently. We can implement a custom cell with Layer Normalization.

In [9]:
class LNSimpleRNNCell(keras.layers.Layer):
    def __init__(self, units, activation="tanh", **kwargs):
        super().__init__(**kwargs)
        self.state_size = units
        self.output_size = units
        self.simple_rnn_cell = keras.layers.SimpleRNNCell(units,
                                                          activation=None)
        self.layer_norm = keras.layers.LayerNormalization()
        self.activation = keras.activations.get(activation)

    def call(self, inputs, states):
        outputs, new_states = self.simple_rnn_cell(inputs, states)
        norm_outputs = self.activation(self.layer_norm(outputs))
        return norm_outputs, [norm_outputs]

model = keras.models.Sequential([
    keras.layers.RNN(LNSimpleRNNCell(20), return_sequences=True,
                     input_shape=[None, 1]),
    keras.layers.RNN(LNSimpleRNNCell(20), return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'ln_simple_rnn_cell', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'ln_simple_rnn_cell_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


### LSTM (Long Short-Term Memory)
The LSTM cell solves the short-term memory problem by maintaining a separate long-term 4state (56$c_{(t)}$) and a short-term state (78$h_{(t)}$). It uses three gates to control information flow:910
1. Forget Gate: Controls what parts of the long-term state should be erased.
2. Input Gate: Controls what parts of the new input should be added to the long-term state.
3. Output Gate: Controls what parts of the long-term state should be read and output as the short-term state.

In [10]:
model = keras.models.Sequential([
    keras.layers.LSTM(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.LSTM(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

### GRU (Gated Recurrent Unit)
A simplified version of LSTM that merges the cell state and hidden state, and combines the forget and input gates into a single update gate. It is computationally more efficient than LSTM.

In [11]:
model = keras.models.Sequential([
    keras.layers.GRU(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

## Using 1D Convolutional Layers (1D ConvNets)
We can use 1D convolutions to process sequences. A 1D filter slides across the time dimension. This allows the network to learn local patterns (like audio snippets) and is very efficient. We often downsample using stride or 1D pooling.

In [12]:
model = keras.models.Sequential([
    keras.layers.Conv1D(filters=20, kernel_size=4, strides=2, padding="valid",
                        input_shape=[None, 1]),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## WaveNet
WaveNet is a powerful architecture composed of a stack of 1D convolutional layers with increasing dilation rates.
- Dilation Rate: The distance between inputs that the filter processes (e.g., rate 2 means skipping every other input).
- By doubling the dilation rate at each layer (1, 2, 4, 8...), the network's receptive field grows exponentially, allowing it to capture extremely long-term patterns efficiently.

In [13]:
model = keras.models.Sequential()
# Explicit input layer
model.add(keras.layers.InputLayer(input_shape=[None, 1]))

# Stack of dilated convolutions
for rate in (1, 2, 4, 8) * 2:
    model.add(keras.layers.Conv1D(filters=20, kernel_size=2, padding="causal",
                                  activation="relu", dilation_rate=rate))

model.add(keras.layers.Conv1D(filters=10, kernel_size=1))
model.compile(loss="mse", optimizer="adam")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Note on "Causal" Padding: This ensures that the convolution output at time $t$ only depends on inputs from time $t$ and earlier, preventing the model from "cheating" by seeing the future.